<h1>Chapter 2 - Generation Models</h1>
<i>Choosing the generation model for your RAG system.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch02_generation/generation.ipynb)

---

This notebook is for Chapter 2 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


In [1]:
%pip install openai==2.21.0 anthropic==0.18.1 python-dotenv==1.0.0 httpx==0.27.0

  Using cached python_dotenv-1.0.0-py3-none-any.whl.metadata (21 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.68.2-py3-none-any.whl.metadata (58 kB)
  Using cached certifi-2026.5.20-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-macosx_11_0_arm64.whl.metadata (9.8 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached huggingface_hub-1.19.0-py3-no

### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [2]:
import os
import sys
from dotenv import load_dotenv

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [4]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

⚠ Running locally. Using ../datasets/ directory


## 1. OpenAI Chat Completions

In [6]:
from openai import OpenAI
import httpx

load_dotenv(override=True)
base_url = os.getenv("AZURE_OPENAI_ENDPOINT")
print("base_url:", base_url)   # kurz prüfen, ob /openai/v1/ dranhängt

def ask_with_context(context, question):
    client = OpenAI(
            api_key=os.getenv("AZURE_OPENAI_KEY"),          # <- statt AZURE_OPENAI_API_KEY
            base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),    # enthält /openai/v1 bereits
            http_client=httpx.Client(),

    )

    messages = [
        {
            "role": "system",
            "content": "Answer based only on the provided context."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion:\n{question}"
        },
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    return response.choices[0].message.content


# Usage
context = "RAG stands for Retrieval-Augmented Generation."
question = "What does RAG stand for?"
answer = ask_with_context(context, question)
print(answer)

base_url: https://ghcp-sws-resource.openai.azure.com/openai/v1
RAG stands for Retrieval-Augmented Generation.


## 2. OpenAI Whisper Speech-to-Text

In [7]:
import os
os.listdir()

['generation.ipynb', 'rag_basics.ipynb', '.env', '.venv', '.idea']

In [11]:
from openai import OpenAI
import os

client = OpenAI(
               api_key=os.getenv("AZURE_OPENAI_KEY"),          # <- statt AZURE_OPENAI_API_KEY
            base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),    # enthält /openai/v1 bereits
            http_client=httpx.Client(),
)

# Determine the correct path based on environment
if IN_COLAB:
    audio_path = "/content/datasets/audio_files/harvard.wav"
else:
    audio_path = "../datasets/audio_files/harvard.wav"

with open(audio_path, "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=audio_file,
    )

print(transcript.text)

NotFoundError: Error code: 404 - {'error': {'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}

## 3. Anthropic Claude Example

In [10]:
client.models.list()

SyncPage[Model](data=[Model(id='dall-e-3-3.0', created=None, object='model', owned_by=None, status='succeeded', capabilities={'fine_tune': False, 'inference': True, 'completion': False, 'chat_completion': False, 'embeddings': False, 'global_fine_tune': False, 'devtier_fine_tune': False}, lifecycle_status='deprecated', deprecation={'inference': 1772582400}, created_at=1691712000), Model(id='dall-e-2-2.0', created=None, object='model', owned_by=None, status='succeeded', capabilities={'fine_tune': False, 'inference': True, 'completion': False, 'chat_completion': False, 'embeddings': False, 'global_fine_tune': False, 'devtier_fine_tune': False}, lifecycle_status='deprecated', deprecation={'inference': 1737936000}, created_at=1713139200), Model(id='tts-001', created=None, object='model', owned_by=None, status='succeeded', capabilities={'fine_tune': False, 'inference': True, 'completion': False, 'chat_completion': False, 'embeddings': False, 'global_fine_tune': False, 'devtier_fine_tune': Fa

In [8]:
from anthropic import Anthropic
import os

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Explain how vector databases work in "
                       "simple terms.",
        }
    ],
)

print(response.content[0].text)

# Vector Databases - Simple Explanation

## The Basic Idea

Think of a vector database like a smart filing system that organizes things by **meaning** rather than exact matches.

## How It Works

**1. Converting to Vectors**
- Everything (text, images, audio) gets converted into lists of numbers called "vectors"
- Similar items get similar numbers
- Example: "dog" and "puppy" would have very similar vectors, while "dog" and "car" would be very different

**2. Storing the Vectors**
- These number lists are stored in a special database
- Each vector represents the "meaning" or characteristics of the original item

**3. Searching by Similarity**
- When you search, your query also becomes a vector
- The database finds vectors that are mathematically "close" to yours
- It returns the most similar items, even if the exact words don't match


## 4. Gemini API Example using the OpenAI SDK

In [9]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

client.models.list()

SyncPage[Model](data=[Model(id='models/gemini-2.5-flash', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Flash'), Model(id='models/gemini-2.5-pro', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Pro'), Model(id='models/gemini-2.0-flash', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash'), Model(id='models/gemini-2.0-flash-001', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash 001'), Model(id='models/gemini-2.0-flash-lite-001', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash-Lite 001'), Model(id='models/gemini-2.0-flash-lite', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash-Lite'), Model(id='models/gemini-2.5-flash-preview-tts', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Flash Preview TTS'), Model(id='models/gemini-2.5-pro-preview-tts', created=None, object='model', owned_by='goo

In [10]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

resp = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
)

print(resp.choices[0].message.content)

The capital of France is **Paris**.


## 5. Deploy local LLMs using Ollama

### Setting up Ollama in Colab

Since Colab runs in a cloud environment, you need to install and run Ollama directly within the Colab instance. The following steps will set up Ollama and pull the required models.

In [11]:
import subprocess
import os

# Install zstd dependency
!sudo apt-get update && sudo apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
os.environ['OLLAMA_HOST'] = '0.0.0.0'
subprocess.Popen(["ollama", "serve"])

print("Ollama server is starting...")

Sudo is disabled by your organization's policy.
'sh' is not recognized as an internal or external command,
operable program or batch file.


Ollama server is starting...


In [12]:
# Give Ollama some time to ensure it's fully started (adjust if needed)
import time
time.sleep(10)

print("Pulling models...")
!ollama pull qwen3:4b
!ollama pull llama2
!ollama pull mistral
!ollama pull codellama

print("Models pulled successfully. You can now re-run the Ollama examples.")

Pulling models...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 3e4cb1417446: 100% ▕██████████████████▏ 2.5 GB                         
pulling 2d54db2b9bb2: 100% ▕██████████████████▏ 1.5 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling e18a783aae55: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 8934d96d3f08: 100% ▕██████████████████▏ 3.8 GB                         
pulling 8c17c2ebb0ea: 100% ▕██████████████████▏ 7.0 

Models pulled successfully. You can now re-run the Ollama examples.


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 3a43f93b78ec: 100% ▕██████████████████▏ 3.8 GB                         
pulling 8c17c2ebb0ea: 100% ▕██████████████████▏ 7.0 KB                         
pulling 590d74a5569b: 100% ▕██████████████████▏ 4.8 KB                         
pulling 2e0493f67d0c: 100% ▕██████████████████▏   59 B                         
pulling 7f6a57943a88: 100% ▕██████████████████▏  120 B                         
pulling 316526ac7323: 100% ▕██████████████████▏  529 B                         
verifying sha256 digest 
writing manifest 
success 


In [2]:
from openai import OpenAI

# Point the client to your local Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama does not require a real key,
                       # but the SDK expects one
)

response = client.chat.completions.create(
    model="qwen3:4b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "What is retrieval augmented generation?"
        },
    ],
)

print(response.choices[0].message.content)

**Retrieval Augmented Generation (RAG)** is an AI technique that **enhances the capabilities of large language models (LLMs)** by **first retrieving relevant external information** from a source (like documents, databases, or knowledge bases) *before* generating responses. This approach combines the strengths of **retrieval systems** (to access precise, up-to-date information) and **generative AI** (to create natural, context-aware responses).  

### Why RAG Matters (The Problem It Solves)
Traditional LLMs (like GPT) are trained on vast historical data but **lack real-time knowledge** and **can hallucinate facts** when answering new questions. RAG solves this by:
1. **Using the latest information** (e.g., company policies, recent research, live data).
2. **Reducing hallucinations** (since responses are grounded in retrieved sources).
3. **Improving accuracy** for domain-specific queries (e.g., medical, legal, technical topics).

---

### How RAG Works (Simplified Step-by-Step)
1. **Use

In [3]:
from openai import OpenAI

models = ["llama2", "mistral", "codellama"]
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

for model in models:
    print(f"\n--- Testing {model} ---")
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": "Explain RAG in one sentence."}
        ]
    )
    print(response.choices[0].message.content)


--- Testing llama2 ---

RAG (Red, Amber, Green) is a visual management technique used to track and monitor progress, status, or quality in a project or process by assigning a color code to each level of progress, with green indicating on-track or completed, amber indicating partially completed or at risk, and red indicating significantly delayed or off-track.

--- Testing mistral ---
 RAG is a rating system, commonly used in project management, to represent the status of a project based on its risk, achievement, and growth factors, where R represents Red for high risk or unfavorable conditions, A represents Amber for moderate risk or uncertain conditions, and G represents Green for low risk or favorable conditions.

--- Testing codellama ---

RAG is a financial concept used to describe the color-coded status of a company's financial performance, with "Red" indicating a decline in performance, "Amber" indicating a moderate improvement, and "Green" indicating a strong performance.


## 6. Pydantic Structured Output

In [13]:
from openai import OpenAI
from pydantic import BaseModel
from datetime import date
from typing import List

class LineItem(BaseModel):
    description: str
    quantity: int
    total: float

class Invoice(BaseModel):
    invoice_number: str
    invoice_date: date
    supplier: str
    items: List[LineItem]
    total_due: float

client = OpenAI(
            api_key=os.getenv("AZURE_OPENAI_KEY"),          # <- statt AZURE_OPENAI_API_KEY
            base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),    # enthält /openai/v1 bereits
            http_client=httpx.Client(),
)

completion = client.chat.completions.parse(
    model="gpt-5.4",
    messages=[
        {"role": "system", "content": "Extract the invoice data from the provided context."},
        {"role": "user", "content": "Invoice #12345, Date: 2024-01-15, Supplier: Tech Corp. Item: Laptop, Qty: 2, Total: $2000. Item: Mouse, Qty: 5, Total: $100. Total Due: $2100"}
    ],
    response_format=Invoice,
)

invoice = completion.choices[0].message.parsed
print(invoice)

invoice_number='12345' invoice_date=datetime.date(2024, 1, 15) supplier='Tech Corp' items=[LineItem(description='Laptop', quantity=2, total=2000.0), LineItem(description='Mouse', quantity=5, total=100.0)] total_due=2100.0
